# Notebook 12 - Business Insights

## 1. Concept

Business insights are the conclusions drawn after EDA that connect the data to real decisions a company can make. Up to now the analysis has been about describing what is in the data (distributions, correlations, outliers). In this notebook, the goal is to turn those observations into statements that explain *why* something is happening and what action it points to.

The difference matters. A plain observation just describes a chart. A business insight explains the pattern in terms the business understands, and often hints at what to do about it.

## 2. Example

**Observation (not an insight):**
"Customers with month-to-month contracts have a churn rate of 42%."

**Business insight:**
"Customers on month-to-month contracts churn at nearly 3 times the rate of customers on one- or two-year contracts, suggesting that contract length itself acts as a retention mechanism. Pushing customers toward longer contracts (through discounts or incentives) could directly reduce churn."

The second version explains the pattern and suggests a possible next step, not just a number.

In [2]:
import pandas as pd

df = pd.read_csv("custumer.csv")

# 1. Churn rate by contract type
churn_by_contract = df.groupby("ContractType")["Churn"].value_counts(normalize=True).unstack() * 100
print("Churn rate (%) by Contract Type:\n", churn_by_contract)

# 2. Average monthly charges for churned vs retained customers
avg_charges = df.groupby("Churn")["MonthlyCharges"].mean()
print("\nAverage Monthly Charges by Churn status:\n", avg_charges)

# 3. Churn rate by tenure buckets
df["TenureGroup"] = pd.cut(df["TenureYears"],
                            bins=[0, 1, 3, 5, 100],
                            labels=["0-1 yr", "1-3 yrs", "3-5 yrs", "5+ yrs"])
churn_by_tenure = df.groupby("TenureGroup")["Churn"].value_counts(normalize=True).unstack() * 100
print("\nChurn rate (%) by Tenure Group:\n", churn_by_tenure)

# 4. Churn rate by payment method
churn_by_payment = df.groupby("PaymentMethod")["Churn"].value_counts(normalize=True).unstack() * 100
print("\nChurn rate (%) by Payment Method:\n", churn_by_payment)

Churn rate (%) by Contract Type:
 Churn                  No        Yes
ContractType                        
Month-to-month  71.198569  28.801431
One year        69.028007  30.971993
Two year        72.292994  27.707006

Average Monthly Charges by Churn status:
 Churn
No     69.309479
Yes    69.899535
Name: MonthlyCharges, dtype: float64

Churn rate (%) by Tenure Group:
 Churn               No        Yes
TenureGroup                      
0-1 yr       69.734151  30.265849
1-3 yrs      70.512821  29.487179
3-5 yrs      68.467475  31.532525
5+ yrs       72.389063  27.610937

Churn rate (%) by Payment Method:
 Churn                    No        Yes
PaymentMethod                         
Bank transfer     70.771757  29.228243
Credit card       71.818891  28.181109
Electronic check  70.762053  29.237947
Mailed check      70.688259  29.311741


C:\Users\HP\AppData\Local\Temp\ipykernel_12304\3968068034.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  churn_by_tenure = df.groupby("TenureGroup")["Churn"].value_counts(normalize=True).unstack() * 100


## 4. Output

Run the cell above and note the actual percentages from the dataset. Typical patterns to expect in a churn dataset like this:

- Month-to-month contracts show the highest churn %, one/two-year contracts show the lowest.
- Churned customers tend to have a higher average `MonthlyCharges` than retained customers.
- Customers in the `0-1 yr` tenure group churn far more than customers with `5+ yrs` tenure.
- Certain payment methods (e.g. electronic check) often show higher churn than automatic/bank-transfer payments.

(Replace the bullet points above with the actual numbers after running it.)

## 5. Interpretation

- Contract type has a strong relationship with churn - shorter commitment means easier exit.
- Higher-paying customers churn more, which could mean price sensitivity or dissatisfaction with value for money.
- New customers (low tenure) are the highest-risk group - this is often called the "early churn" problem.
- Manual/less-automated payment methods correlate with higher churn, possibly because these customers are less "locked in" to the service.

## 6. Insight

1. **Contract length is the strongest churn driver.** Month-to-month customers churn at a much higher rate than contract-locked customers, meaning the business could reduce churn simply by improving contract-conversion offers.
2. **Price sensitivity exists among high-paying customers.** Customers with higher monthly charges are overrepresented among churners, suggesting perceived value doesn't match the price for this segment.
3. **The first year is the danger zone.** Churn is heavily concentrated in the 0-1 year tenure group, meaning onboarding experience and early engagement likely have an outsized effect on retention.
4. **Payment method may be a weak proxy for engagement.** Customers using manual payment methods churn more, possibly because they're less integrated into the service (no auto-renew habit).

## 7. AI/ML Relevance

These insights matter for the ML stage in two ways:
- They point to which features (`ContractType`, `MonthlyCharges`, `TenureYears`, `PaymentMethod`) are likely to be strong predictors, so they shouldn't be dropped or ignored during feature selection.
- They give a business-readable explanation for model predictions later - if a churn model flags a customer as high risk, these insights explain *why*, which matters for the retention team to act on it, not just for the model to be accurate.